In [1]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS, LOG_PATH


# 0. Read the data

In [3]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

In [4]:
METHOD = "ghmm"

## 0.1. make result dicts

In [5]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

## 0.2 Read decoy data

In [6]:
per_allele_decoy = dict()

for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    DECOY_DATA_PATH = "/home/denklewer/Data/decoy/sampled_decoy_data_{}_class_{}.csv".format(organism, class_to_use)
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    decoy_df['allele'] = allele_name
    
    per_allele_decoy[allele_name] = decoy_df.copy()
[len(decoy_df) for decoy_df in per_allele_decoy.values()]

[200000, 200000, 200000, 200000, 200000]

# 1. Predict with ghmm

In [7]:
import numpy as np
import pandas as pd
import ghmm
import os
import logging
import sys
from scipy.version import full_version as spversion

LOG_DIR = LOG_PATH + 'ghmm/'

if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)
    
logging.basicConfig(
    filename='{}{}_train.log'.format(LOG_DIR,METHOD), 
    filemode='a',
    encoding='utf-8', 
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)-1s: %(message)s',
            datefmt='%d-%m-%y %H:%M:%S') 
logger = logging.getLogger("GHMM")
logger.setLevel(logging.INFO)

In [8]:
logger.info('Python    :{}'.format(sys.version))
logger.info('Numpy     :{}'.format(np.__version__))
logger.info("Scipy     :{}".format(spversion))
logger.info('Pandas    :{}'.format(pd.__version__))

logger.info("GHMM:      0.9 (manually tracked)") # no such thing as ghmm.__version__, and not under pip
logger.info('Workdir   :{}'.format(os.getcwd()))
logger.setLevel(logging.CRITICAL)

GHMM <ipython-input-8-ef26eef7187c>:1 - Python    :2.7.18 |Anaconda, Inc.| (default, Jun  4 2021, 14:47:46) 
[GCC 7.3.0]
GHMM <ipython-input-8-ef26eef7187c>:2 - Numpy     :1.16.6
GHMM <ipython-input-8-ef26eef7187c>:3 - Scipy     :1.2.3
GHMM <ipython-input-8-ef26eef7187c>:4 - Pandas    :0.24.2
GHMM <ipython-input-8-ef26eef7187c>:6 - GHMM:      0.9 (manually tracked)
GHMM <ipython-input-8-ef26eef7187c>:7 - Workdir   :/home/denklewer/Projects/MHC_predictor/experiments/schreiber_peptides_full_processing


In [9]:
#change allele naming
TARGET_ALLELES = dict()

for allele_name in ALLELES:
    target_allele_name = normalize_allele_name(allele_name).replace("*", "").replace("-", "").lower()
    TARGET_ALLELES[allele_name] = target_allele_name


TARGET_ALLELES

{'H-2-Dd': 'h2dd',
 'H-2-IAd': 'h2iad',
 'H-2-IEd': 'h2ied',
 'H-2-Kd': 'h2kd',
 'H-2-Ld': 'h2ld'}

In [12]:
MODEL_PATH = '/home/denklewer/Models/Katya_ghmm/'
#MODEL_PATH = '/home/denklewer/Models/ghmm/'
#MODEL_PATH = '../trained_models/old_ghmm/'
#supported_alleles = ['h2iab', 'h2iad', 'iag7', 'h2iak', 'h2iaq','h2ias', 'h2iau', 'h2ied', 'h2iek' ]
supported_alleles = ['h2ld', 'h2kd', 'h2kb', 'h2iek', 'h2ied',
                     'h2iau', 'h2ias', 'h2iaq', 'h2iak', 'h2iag7',
                     'h2iad', 'h2iab', 'h2dd', 'h2db']
#
        
        
AminoAcidsZ = ghmm.Alphabet(ghmm.AminoAcids.listOfCharacters + ['Z'])

def prepareHmmSequenceSets(peptides, stopEmission=True):
    if stopEmission:
        peptideList = [str(p) + 'Z' for p in peptides]
    else:
        peptideList = [str(p) for p in peptides]
    
    # need to split in blocks in case there are more than 1.5M peptides (ghmm limitation)
    ghmmMaxSequencesPerSet = 1500000
    lenPeptides = len(peptides)
    numFullBlocks = lenPeptides // ghmmMaxSequencesPerSet
    lenLastBlock = lenPeptides % ghmmMaxSequencesPerSet
    sequenceSets = []
    
    # full blocks (if any)
    for i in range(0, numFullBlocks):
        rangeStart = i * ghmmMaxSequencesPerSet
        rangeEnd = rangeStart + ghmmMaxSequencesPerSet
        sequenceBlock = ghmm.SequenceSet(
            AminoAcidsZ,
            [list(p) for p in peptideList[rangeStart:rangeEnd]]
        )
        sequenceSets.append(sequenceBlock)
    rangeStart = numFullBlocks * ghmmMaxSequencesPerSet
    rangeEnd = lenPeptides
    if (lenLastBlock > 0):
        sequenceBlock = ghmm.SequenceSet(
                AminoAcidsZ,
                [list(p) for p in peptideList[rangeStart:rangeEnd]]
            )
        sequenceSets.append(sequenceBlock)
    return sequenceSets

def predictHmmPeptidesRaw(hmmModelFile, sequenceSets):
    """Predict -10logp values using a given HMM model for binders and non-binders.
       Results will contain -inf values
    """
    hmm = ghmm.HMMOpen(hmmModelFile)

    return np.concatenate(
        [-10 * np.array(hmm.loglikelihoods(s)) for s in sequenceSets]
    )


def predict_with_ghmm(per_allele_df):
    per_allele_predictions = dict()
    for allele_name in ALLELES:
        print(allele_name)
        target_allele = TARGET_ALLELES[allele_name]
        df_val = per_allele_df[allele_name].copy()
        if target_allele in supported_alleles:
            sequenceSet = prepareHmmSequenceSets(df_val.peptide)
            print("Predict with model")
            print("Binders")
            df_val['HMMB'] = predictHmmPeptidesRaw('{}{}-splitwithdecoy-binders.xml'.format(MODEL_PATH, target_allele),sequenceSet)

            columnsHMM = ['HMMB']
            df_val.loc[:, columnsHMM] = df_val.loc[:, columnsHMM].div(df_val.length, axis=0)
            df_val = df_val.replace(np.inf, 10000.0)
        else:
            print("Allele {} is not supported, create dummy predictions".format(target_allele))
            df_val['HMMB'] = np.nan
        per_allele_predictions[allele_name] = df_val
        
    
    return per_allele_predictions

def update_scores_with_percentiles(per_allele_scores, per_allele_ref_scores, score_name):
    for allele_name in per_allele_scores.keys():
        df = per_allele_scores[allele_name].copy()
        ref_df =  per_allele_ref_scores[allele_name].copy()
        scores = df[score_name]
        ref_scores = ref_df[score_name]
        ref_scores = ref_scores.sort_values()
        pct_increment = 1./len(ref_scores)
        percentile_column_name = 'percentile_rank'
        percentile_ranks = (0.5 + 0.5*ref_scores.searchsorted(scores, side='left') + 0.5 * ref_scores.searchsorted(scores, side='right'))*pct_increment
        percentile_ranks = percentile_ranks*100
        per_allele_scores[allele_name][percentile_column_name] = percentile_ranks
    return per_allele_scores
    

# Calculate for decoy

In [13]:
%%time

print("Decoy")
per_allele_decoy_result = predict_with_ghmm(per_allele_decoy)

Decoy
H-2-Dd
Predict with model
Binders
H-2-IEd
Predict with model
Binders
H-2-Ld
Predict with model
Binders
H-2-Kd
Predict with model
Binders
H-2-IAd
Predict with model
Binders
CPU times: user 42.5 s, sys: 335 ms, total: 42.8 s
Wall time: 42.7 s


In [14]:
%%time
print("--mt")
per_allele_mt_predictions = predict_with_ghmm(per_allele_mt_df)
per_allele_mt_predictions = update_scores_with_percentiles(per_allele_mt_predictions, per_allele_decoy_result, 'HMMB' )

per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions  
print("--wt")
per_allele_wt_predictions = predict_with_ghmm(per_allele_wt_df)
per_allele_wt_predictions = update_scores_with_percentiles(per_allele_wt_predictions, per_allele_decoy_result, 'HMMB' )

per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions  


--mt
H-2-Dd
Predict with model
Binders
H-2-IEd
Predict with model
Binders
H-2-Ld
Predict with model
Binders
H-2-Kd
Predict with model
Binders
H-2-IAd
Predict with model
Binders
--wt
H-2-Dd
Predict with model
Binders
H-2-IEd
Predict with model
Binders
H-2-Ld
Predict with model
Binders
H-2-Kd
Predict with model
Binders
H-2-IAd
Predict with model
Binders
CPU times: user 16.2 s, sys: 444 ms, total: 16.6 s
Wall time: 16.2 s


# 2. Join predictions

In [15]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity', 'percentile_rank'],
    'ghmm':['source_sequence_name', 'HMMB', 'percentile_rank']

}
def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df


def getPredictionResult(total_results, allele_name, method):
    df = total_results[method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df
# def getPredictionResult(total_results, region, allele_name, method):
#     df = total_results[region][method][allele_name][per_method_columns[method]].copy()
#     df = rename_columns(df, method)
#     return df


In [16]:
def joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions):
    per_allele_result = dict()
    methods = list(per_method_per_allele_mt_predictions.keys())
    for allele_name in ALLELES:
        print("allele", allele_name)
        df_source_wt = per_allele_wt_df[allele_name].copy()
        df_source_mt = per_allele_mt_df[allele_name].copy()

        for method in methods:
            print("method", method)
            df_method_wt = getPredictionResult(per_method_per_allele_wt_predictions, allele_name, method)
            df_method_mt = getPredictionResult(per_method_per_allele_mt_predictions, allele_name, method)
            df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
            df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
        leave_columns = [ 'source_sequence_name', 'allele', 'length']
        df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
        df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
        # rename id back 


        df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
        per_allele_result[allele_name] = df_result
    return per_allele_result
                

              

In [17]:
per_allele_result = joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions)

('allele', 'H-2-Dd')
('method', 'ghmm')
('allele', 'H-2-IEd')
('method', 'ghmm')
('allele', 'H-2-Ld')
('method', 'ghmm')
('allele', 'H-2-Kd')
('method', 'ghmm')
('allele', 'H-2-IAd')
('method', 'ghmm')


# 5. For now just write to files

In [18]:
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + 'predictions/{}/'.format(METHOD)

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [19]:
# for decoy_key in DECOY_PREFIXES:

for allele_name in ALLELES:
    total_df = per_allele_result[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}{}_predictions_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,allele_name),sep=";", index=False)

In [21]:
per_allele_result[ALLELES[0]]

,source_sequence_name,allele,peptide_mt,length,ghmm_HMMB_mt,ghmm_percentile_rank_mt,peptide_wt,ghmm_HMMB_wt,ghmm_percentile_rank_wt
0,seq124,H-2-Dd,SALSSLLL,8,26.725338,0.01175,SALSILLL,26.319126,0.00875
1,seq194,H-2-Dd,FVTAVAETQ,9,27.150790,0.01525,SVTAVAETQ,35.636783,0.24975
2,seq329,H-2-Dd,FVTAVAET,8,28.695724,0.03225,SVTAVAET,38.242467,0.37575
3,seq401,H-2-Dd,GGACLLLL,8,29.212166,0.04025,GGDCLLLL,10000.000000,50.74275
4,seq431,H-2-Dd,DVINSCTQ,8,29.299179,0.04125,DVINSRTQ,58.619317,1.18075
5,seq510,H-2-Dd,FVAVVITI,8,29.769634,0.04775,FVAAVITI,29.661047,0.04625
6,seq543,H-2-Dd,FAVLAVVL,8,29.853595,0.04975,FAVLVVVL,10000.000000,50.74275
7,seq605,H-2-Dd,SFVAVVITI,9,29.733428,0.04725,SFVAAVITI,29.636907,0.04575
8,seq658,H-2-Dd,VAAAEYSGL,9,29.997092,0.05225,VAAAEYSGP,10000.000000,50.74275
9,seq659,H-2-Dd,AAAPAAALSAA,11,29.493679,0.04425,PAAPAAALSAA,10000.000000,50.74275
